# CLIP4CAD-GFA v4.4 Evaluation

This notebook evaluates the trained GFA v4.4 model with **Topology-Aware Encoding**.

## Key Innovations in v4.4
1. **Topology Message Passing** - Face↔Edge information exchange via edge_to_faces graph
2. **Hierarchical Pattern Aggregation** - BFS-level grouping for semantic patterns
3. **Multi-Source Query Generation** - Attend to tokens + level summaries + global
4. **Contrastive Query Loss** - Instance-level Q_self → T_feat matching

## Evaluation Metrics
1. **Retrieval metrics**: Text→BRep, Text→PC, PC→BRep R@K
2. **Self-grounding quality**: Cosine similarity between guided and self embeddings
3. **Query alignment**: Cosine similarity between T_feat and Q_self (KEY METRIC!)
4. **Text-free inference**: Compare self-grounding retrieval with text-guided
5. **Level summary analysis**: Hierarchical pattern representation quality

## Expected Results (v4.4 with Topology)
| Metric | v4 | v4.4 Target |
|--------|-----|-------------|
| Self-cos BRep | 0.18 | **0.65-0.80** |
| Query alignment | 0.78 | **≥0.85** |
| Text→BRep R@1 (self) | ~8% | **40-60%** |
| Gap (guided - self) | 63% | **<15%** |

In [ ]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import gc
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast
from tqdm.auto import tqdm
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path
import json
from datetime import datetime

# Clean memory
gc.collect()
torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Configuration

# ═══════════════════════════════════════════════════════════════════════════════
# PATHS
# ═══════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"

# Model checkpoint
CHECKPOINT_PATH = Path("../outputs/gfa_v4_4/checkpoint_best.pt")
# Alternative: use final model
# CHECKPOINT_PATH = Path("../outputs/gfa_v4_4/gfa_v4_4_final.pt")

# Data files
BREP_VAL = EMBEDDINGS_DIR / "val_brep_autobrep.h5"
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

# Output
OUTPUT_DIR = Path("../outputs/gfa_v4_4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Evaluation settings
BATCH_SIZE = 64

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")

In [ ]:
# Cell 3: Load Model

from clip4cad.models import CLIP4CAD_GFA_v44, GFAv44Config

# Create model with same config as training
config = GFAv44Config(
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    d_unified=256,
    d_proj=128,
    d_ground=128,
    num_slots=12,
    num_detail_queries=8,
    num_heads=8,
    # Topology encoder
    num_msg_layers=3,
    num_brep_tf_layers=4,
    max_bfs_levels=32,
    # Pattern aggregator
    num_pattern_levels=10,
    # Query generator
    num_query_layers=4,
    num_pc_query_layers=2,
    dropout=0.0,  # Disable dropout for evaluation
)

model = CLIP4CAD_GFA_v44(config).to(device)

# Load weights
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Set conditioning dropout to 1.0 (no hints) for true text-free evaluation
model.set_cond_dropout(1.0, 1.0)

print(f"Loaded model from {CHECKPOINT_PATH}")
print(f"Epoch: {checkpoint.get('epoch', 'N/A')}")
print(f"Best self-cosine: {checkpoint.get('best_self_cosine', 'N/A')}")
print(f"Model parameters: {model.count_parameters():,}")

In [ ]:
# Cell 4: Dataset with Topology Fields

class GFAv44EvalDataset(Dataset):
    """
    Evaluation dataset with AutoBrep topology features.
    Optimized for evaluation (no data augmentation, memory-mapped).
    """
    
    def __init__(
        self,
        brep_file: str,
        pc_file: str,
        text_file: str,
        split: str = 'val',
        load_to_memory: bool = False  # Memory-mapped for eval
    ):
        self.split = split
        self.load_to_memory = load_to_memory
        
        # Open BRep file (with topology)
        print(f"Loading BRep features from {brep_file}...")
        self.brep_h5 = h5py.File(brep_file, 'r')
        self.brep_uids = [u.decode() if isinstance(u, bytes) else u 
                         for u in self.brep_h5['uids'][:]]
        self.n_samples = len(self.brep_uids)
        
        # Build UID index
        self.uid_to_idx = {uid: i for i, uid in enumerate(self.brep_uids)}
        
        # Open PC file
        print(f"Loading PC features from {pc_file}...")
        self.pc_h5 = h5py.File(pc_file, 'r')
        self.pc_uids = [u.decode() if isinstance(u, bytes) else u 
                       for u in self.pc_h5[split]['uids'][:]]
        self.pc_uid_to_idx = {uid: i for i, uid in enumerate(self.pc_uids)}
        
        # Check PC structure
        if 'local_features' in self.pc_h5[split]:
            self.pc_format = 'split'  # local_features + global_features
        else:
            self.pc_format = 'concat'  # single pc_features array
        
        # Open Text file
        print(f"Loading Text features from {text_file}...")
        self.text_h5 = h5py.File(text_file, 'r')
        self.text_uids = [u.decode() if isinstance(u, bytes) else u 
                         for u in self.text_h5[split]['uids'][:]]
        self.text_uid_to_idx = {uid: i for i, uid in enumerate(self.text_uids)}
        
        print(f"Dataset ready: {self.n_samples} samples (format: {self.pc_format})")
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        uid = self.brep_uids[idx]
        
        # BRep features (from HDF5)
        face_feat = torch.from_numpy(self.brep_h5['face_features'][idx]).float()
        edge_feat = torch.from_numpy(self.brep_h5['edge_features'][idx]).float()
        face_mask = torch.from_numpy(self.brep_h5['face_masks'][idx]).bool()
        edge_mask = torch.from_numpy(self.brep_h5['edge_masks'][idx]).bool()
        edge_to_faces = torch.from_numpy(self.brep_h5['edge_to_faces'][idx]).long()
        face_centroids = torch.from_numpy(self.brep_h5['face_centroids'][idx]).float()
        bfs_level = torch.from_numpy(self.brep_h5['bfs_level'][idx]).long()
        
        # Optional topology fields (use placeholders if missing)
        if 'face_normals' in self.brep_h5:
            face_normals = torch.from_numpy(self.brep_h5['face_normals'][idx]).float()
        else:
            face_normals = torch.zeros_like(face_centroids)
        
        if 'face_areas' in self.brep_h5:
            face_areas = torch.from_numpy(self.brep_h5['face_areas'][idx]).float()
        else:
            face_areas = torch.ones(face_feat.shape[0])
        
        if 'edge_midpoints' in self.brep_h5:
            edge_midpoints = torch.from_numpy(self.brep_h5['edge_midpoints'][idx]).float()
        else:
            edge_midpoints = torch.zeros((edge_feat.shape[0], 3))
        
        if 'edge_lengths' in self.brep_h5:
            edge_lengths = torch.from_numpy(self.brep_h5['edge_lengths'][idx]).float()
        else:
            edge_lengths = torch.ones(edge_feat.shape[0])
        
        # PC features
        pc_idx = self.pc_uid_to_idx.get(uid, 0)
        if self.pc_format == 'split':
            pc_local = torch.from_numpy(self.pc_h5[self.split]['local_features'][pc_idx]).float()
            pc_global = torch.from_numpy(self.pc_h5[self.split]['global_features'][pc_idx]).float()
        else:
            pc_feat = torch.from_numpy(self.pc_h5[self.split]['pc_features'][pc_idx]).float()
            pc_local = pc_feat[:32]   # First 32 are local
            pc_global = pc_feat[32:]  # Rest are global (16)
        
        # Text features
        text_idx = self.text_uid_to_idx.get(uid, 0)
        text_feat = torch.from_numpy(self.text_h5[self.split]['embeddings'][text_idx]).float()
        
        return {
            'face_features': face_feat,
            'edge_features': edge_feat,
            'face_mask': face_mask,
            'edge_mask': edge_mask,
            'edge_to_faces': edge_to_faces,
            'face_centroids': face_centroids,
            'face_normals': face_normals,
            'face_areas': face_areas,
            'bfs_level': bfs_level,
            'edge_midpoints': edge_midpoints,
            'edge_lengths': edge_lengths,
            'pc_local_features': pc_local,
            'pc_global_features': pc_global,
            'text_features': text_feat,
            'uid': uid,
        }
    
    def close(self):
        self.brep_h5.close()
        self.pc_h5.close()
        self.text_h5.close()


def collate_fn(batch):
    """Collate with variable-size padding for topology fields."""
    B = len(batch)
    
    # Get max sizes
    max_faces = max(b['face_features'].shape[0] for b in batch)
    max_edges = max(b['edge_features'].shape[0] for b in batch)
    d_face = batch[0]['face_features'].shape[-1]
    d_edge = batch[0]['edge_features'].shape[-1]
    
    # Initialize tensors
    collated = {
        'face_features': torch.zeros(B, max_faces, d_face),
        'edge_features': torch.zeros(B, max_edges, d_edge),
        'face_mask': torch.zeros(B, max_faces, dtype=torch.bool),
        'edge_mask': torch.zeros(B, max_edges, dtype=torch.bool),
        'edge_to_faces': torch.full((B, max_edges, 2), -1, dtype=torch.long),
        'face_centroids': torch.zeros(B, max_faces, 3),
        'face_normals': torch.zeros(B, max_faces, 3),
        'face_areas': torch.zeros(B, max_faces),
        'bfs_level': torch.zeros(B, max_faces, dtype=torch.long),
        'edge_midpoints': torch.zeros(B, max_edges, 3),
        'edge_lengths': torch.zeros(B, max_edges),
        'pc_local_features': torch.stack([b['pc_local_features'] for b in batch]),
        'pc_global_features': torch.stack([b['pc_global_features'] for b in batch]),
        'text_features': torch.stack([b['text_features'] for b in batch]),
    }
    
    # Fill in
    for i, b in enumerate(batch):
        n_f = b['face_features'].shape[0]
        n_e = b['edge_features'].shape[0]
        
        collated['face_features'][i, :n_f] = b['face_features']
        collated['edge_features'][i, :n_e] = b['edge_features']
        collated['face_mask'][i, :n_f] = b['face_mask']
        collated['edge_mask'][i, :n_e] = b['edge_mask']
        collated['edge_to_faces'][i, :n_e] = b['edge_to_faces']
        collated['face_centroids'][i, :n_f] = b['face_centroids']
        collated['face_normals'][i, :n_f] = b['face_normals']
        collated['face_areas'][i, :n_f] = b['face_areas']
        collated['bfs_level'][i, :n_f] = b['bfs_level']
        collated['edge_midpoints'][i, :n_e] = b['edge_midpoints']
        collated['edge_lengths'][i, :n_e] = b['edge_lengths']
    
    return collated

In [ ]:
# Cell 5: Load Validation Data
gc.collect()
torch.cuda.empty_cache()

print("Loading validation dataset...")
print("=" * 60)

val_dataset = GFAv44EvalDataset(
    brep_file=str(BREP_VAL),
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    split='val',
    load_to_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=True
)

print(f"\nValidation samples: {len(val_dataset):,}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# Cell 6: Encode All Samples

@torch.no_grad()
def encode_all_samples(model, loader):
    """
    Encode all samples and return embeddings plus query features.
    
    v4.4 specific: Also captures level_summaries for topology analysis.
    """
    model.eval()
    
    # Embeddings
    z_brep_guided_all = []
    z_pc_guided_all = []
    z_text_all = []
    z_brep_self_all = []
    z_pc_self_all = []
    
    # Query features
    T_feat_all = []
    Q_brep_self_all = []
    Q_pc_self_all = []
    conf_text_all = []
    
    # Topology features (v4.4 specific)
    level_summaries_all = []
    global_summary_all = []
    
    for batch in tqdm(loader, desc="Encoding"):
        with autocast():
            outputs = model(batch)
        
        # Embeddings (convert to float32 for accurate metrics)
        z_brep_guided_all.append(F.normalize(outputs['z_brep'].float(), dim=-1).cpu())
        z_pc_guided_all.append(F.normalize(outputs['z_pc'].float(), dim=-1).cpu())
        z_text_all.append(F.normalize(outputs['z_text'].float(), dim=-1).cpu())
        z_brep_self_all.append(F.normalize(outputs['z_brep_self'].float(), dim=-1).cpu())
        z_pc_self_all.append(F.normalize(outputs['z_pc_self'].float(), dim=-1).cpu())
        
        # Query features
        if 'T_feat' in outputs:
            T_feat_all.append(outputs['T_feat'].float().cpu())
        if 'Q_brep_self' in outputs:
            Q_brep_self_all.append(outputs['Q_brep_self'].float().cpu())
        if 'Q_pc_self' in outputs:
            Q_pc_self_all.append(outputs['Q_pc_self'].float().cpu())
        if 'confidence' in outputs:
            conf_text_all.append(outputs['confidence'].float().cpu())
        
        # Topology features (v4.4 specific)
        if 'level_summaries' in outputs:
            level_summaries_all.append(outputs['level_summaries'].float().cpu())
        if 'global_summary' in outputs:
            global_summary_all.append(outputs['global_summary'].float().cpu())
    
    results = {
        'z_brep_guided': torch.cat(z_brep_guided_all),
        'z_pc_guided': torch.cat(z_pc_guided_all),
        'z_text': torch.cat(z_text_all),
        'z_brep_self': torch.cat(z_brep_self_all),
        'z_pc_self': torch.cat(z_pc_self_all),
    }
    
    # Add query features if available
    if T_feat_all:
        results['T_feat'] = torch.cat(T_feat_all)
    if Q_brep_self_all:
        results['Q_brep_self'] = torch.cat(Q_brep_self_all)
    if Q_pc_self_all:
        results['Q_pc_self'] = torch.cat(Q_pc_self_all)
    if conf_text_all:
        results['conf_text'] = torch.cat(conf_text_all)
    
    # Add topology features (v4.4 specific)
    if level_summaries_all:
        results['level_summaries'] = torch.cat(level_summaries_all)
    if global_summary_all:
        results['global_summary'] = torch.cat(global_summary_all)
    
    return results

# Encode all samples
print("Encoding validation set...")
embeddings = encode_all_samples(model, val_loader)

print(f"\nEmbedding shapes:")
print(f"  z_brep_guided: {embeddings['z_brep_guided'].shape}")
print(f"  z_brep_self: {embeddings['z_brep_self'].shape}")
print(f"  z_pc_guided: {embeddings['z_pc_guided'].shape}")
print(f"  z_text: {embeddings['z_text'].shape}")
if 'T_feat' in embeddings:
    print(f"  T_feat: {embeddings['T_feat'].shape}")
if 'Q_brep_self' in embeddings:
    print(f"  Q_brep_self: {embeddings['Q_brep_self'].shape}")
if 'level_summaries' in embeddings:
    print(f"  level_summaries: {embeddings['level_summaries'].shape}")

In [ ]:
# Cell 7: Compute Retrieval Metrics

def compute_retrieval_metrics(query_emb, gallery_emb, k_values=[1, 5, 10]):
    """Compute R@K retrieval metrics."""
    sim = query_emb @ gallery_emb.T
    rankings = torch.argsort(sim, dim=1, descending=True)
    N = query_emb.shape[0]
    labels = torch.arange(N)
    
    results = {}
    for k in k_values:
        top_k = rankings[:, :k]
        correct = (top_k == labels.unsqueeze(1)).any(dim=1)
        results[f'R@{k}'] = correct.float().mean().item() * 100
    
    ranks = (rankings == labels.unsqueeze(1)).nonzero()[:, 1] + 1
    results['MRR'] = (1.0 / ranks.float()).mean().item() * 100
    results['MedR'] = ranks.float().median().item()
    
    return results

# Extract embeddings
z_brep_guided = embeddings['z_brep_guided']
z_pc_guided = embeddings['z_pc_guided']
z_text = embeddings['z_text']
z_brep_self = embeddings['z_brep_self']
z_pc_self = embeddings['z_pc_self']

print("=" * 70)
print("RETRIEVAL RESULTS (Text-Guided)")
print("=" * 70)

# Text → BRep (Guided)
text_brep_guided = compute_retrieval_metrics(z_text, z_brep_guided)
print(f"\nText → BRep (Guided):")
for k, v in text_brep_guided.items():
    print(f"  {k}: {v:.2f}{'%' if 'R@' in k or k == 'MRR' else ''}")

# Text → PC (Guided)
text_pc_guided = compute_retrieval_metrics(z_text, z_pc_guided)
print(f"\nText → PC (Guided):")
for k, v in text_pc_guided.items():
    print(f"  {k}: {v:.2f}{'%' if 'R@' in k or k == 'MRR' else ''}")

# PC → BRep (Guided)
pc_brep_guided = compute_retrieval_metrics(z_pc_guided, z_brep_guided)
print(f"\nPC → BRep (Guided):")
for k, v in pc_brep_guided.items():
    print(f"  {k}: {v:.2f}{'%' if 'R@' in k or k == 'MRR' else ''}")

In [ ]:
# Cell 8: Query Alignment Analysis (KEY METRIC for v4.4!)

print("=" * 70)
print("QUERY ALIGNMENT ANALYSIS (v4.4 Key Metric)")
print("=" * 70)

query_align_brep = 0.0
query_align_pc = 0.0

if 'T_feat' in embeddings and 'Q_brep_self' in embeddings:
    T_feat = embeddings['T_feat']
    Q_brep_self = embeddings['Q_brep_self']
    Q_pc_self = embeddings.get('Q_pc_self')
    conf_text = embeddings.get('conf_text')
    
    # Normalize for cosine similarity
    T_feat_norm = F.normalize(T_feat, dim=-1)
    Q_brep_norm = F.normalize(Q_brep_self, dim=-1)
    
    # Per-slot cosine similarity
    cos_brep = (T_feat_norm * Q_brep_norm).sum(dim=-1)  # (N, K)
    
    # Weighted by confidence if available
    if conf_text is not None:
        weighted_cos_brep = (cos_brep * conf_text).sum(dim=1) / (conf_text.sum(dim=1) + 1e-8)
        query_align_brep = weighted_cos_brep.mean().item()
    else:
        query_align_brep = cos_brep.mean().item()
    
    print(f"\nBRep Query Alignment: {query_align_brep:.4f}")
    print(f"  Per-slot mean: {cos_brep.mean().item():.4f}")
    print(f"  Per-slot min: {cos_brep.min().item():.4f}")
    print(f"  Per-slot max: {cos_brep.max().item():.4f}")
    print(f"  Per-slot std: {cos_brep.std().item():.4f}")
    
    # Per-slot breakdown
    print(f"\n  Per-slot averages:")
    for i in range(cos_brep.shape[1]):
        slot_mean = cos_brep[:, i].mean().item()
        conf_mean = conf_text[:, i].mean().item() if conf_text is not None else 1.0
        print(f"    Slot {i}: cos={slot_mean:.3f}, conf={conf_mean:.3f}")
    
    if Q_pc_self is not None:
        Q_pc_norm = F.normalize(Q_pc_self, dim=-1)
        cos_pc = (T_feat_norm * Q_pc_norm).sum(dim=-1)
        if conf_text is not None:
            weighted_cos_pc = (cos_pc * conf_text).sum(dim=1) / (conf_text.sum(dim=1) + 1e-8)
            query_align_pc = weighted_cos_pc.mean().item()
        else:
            query_align_pc = cos_pc.mean().item()
        print(f"\nPC Query Alignment: {query_align_pc:.4f}")
    
    # Assessment
    print("\n" + "-" * 50)
    if query_align_brep >= 0.85:
        print("✓ EXCELLENT: Query alignment achieves v4.4 target (≥ 0.85)")
        print("  Topology encoding successfully captures semantic features!")
    elif query_align_brep >= 0.70:
        print("○ GOOD: Query alignment improving (0.70-0.85)")
    elif query_align_brep >= 0.50:
        print("△ MODERATE: Query alignment developing (0.50-0.70)")
    else:
        print(f"✗ NEEDS WORK: Query alignment below target (current: {query_align_brep:.4f}, target: ≥ 0.85)")
else:
    print("\nQuery features not available in model outputs.")

In [ ]:
# Cell 9: Self-Grounding Quality

print("=" * 70)
print("SELF-GROUNDING QUALITY")
print("=" * 70)

# Cosine similarity between guided and self embeddings
cos_brep = (z_brep_guided * z_brep_self).sum(dim=-1).mean().item()
cos_pc = (z_pc_guided * z_pc_self).sum(dim=-1).mean().item()

print(f"\nBRep Guided-Self Cosine: {cos_brep:.4f}")
print(f"PC Guided-Self Cosine: {cos_pc:.4f}")

# Distribution
cos_brep_all = (z_brep_guided * z_brep_self).sum(dim=-1)
cos_pc_all = (z_pc_guided * z_pc_self).sum(dim=-1)

print(f"\nBRep Cosine Distribution:")
print(f"  Min: {cos_brep_all.min().item():.4f}")
print(f"  Max: {cos_brep_all.max().item():.4f}")
print(f"  Std: {cos_brep_all.std().item():.4f}")
print(f"  Median: {cos_brep_all.median().item():.4f}")

print(f"\nPC Cosine Distribution:")
print(f"  Min: {cos_pc_all.min().item():.4f}")
print(f"  Max: {cos_pc_all.max().item():.4f}")
print(f"  Std: {cos_pc_all.std().item():.4f}")
print(f"  Median: {cos_pc_all.median().item():.4f}")

# Assessment (v4.4 targets: 0.65-0.80)
print("\n" + "-" * 50)
if cos_brep >= 0.80:
    print("✓ EXCELLENT: Self-grounding exceeds v4.4 target (≥ 0.80)")
elif cos_brep >= 0.65:
    print("✓ GOOD: Self-grounding meets v4.4 target range (0.65-0.80)")
elif cos_brep >= 0.50:
    print("○ MODERATE: Self-grounding improving (0.50-0.65)")
elif cos_brep >= 0.30:
    print(f"△ DEVELOPING: Self-grounding below target (current: {cos_brep:.4f}, target: ≥ 0.65)")
else:
    print(f"✗ NEEDS WORK: Self-grounding below target (current: {cos_brep:.4f}, target: ≥ 0.65)")

In [ ]:
# Cell 10: Text-Free Inference (Self-Grounding Retrieval)

print("=" * 70)
print("TEXT-FREE INFERENCE (Self-Grounding)")
print("=" * 70)

# Text → BRep with self-grounded embeddings
text_brep_self = compute_retrieval_metrics(z_text, z_brep_self)
print(f"\nText → BRep (Self-Grounding):")
for k, v in text_brep_self.items():
    print(f"  {k}: {v:.2f}{'%' if 'R@' in k or k == 'MRR' else ''}")

# Text → PC with self-grounded embeddings
text_pc_self = compute_retrieval_metrics(z_text, z_pc_self)
print(f"\nText → PC (Self-Grounding):")
for k, v in text_pc_self.items():
    print(f"  {k}: {v:.2f}{'%' if 'R@' in k or k == 'MRR' else ''}")

# PC → BRep with self-grounded embeddings
pc_brep_self = compute_retrieval_metrics(z_pc_self, z_brep_self)
print(f"\nPC → BRep (Self-Grounding):")
for k, v in pc_brep_self.items():
    print(f"  {k}: {v:.2f}{'%' if 'R@' in k or k == 'MRR' else ''}")

# Compare with guided
print("\n" + "-" * 50)
print("Comparison: Self vs Guided")
print("-" * 50)

r1_brep_diff = text_brep_self['R@1'] - text_brep_guided['R@1']
r1_pc_diff = text_pc_self['R@1'] - text_pc_guided['R@1']

print(f"Text→BRep R@1: Guided={text_brep_guided['R@1']:.2f}%, Self={text_brep_self['R@1']:.2f}%, Gap={abs(r1_brep_diff):.2f}%")
print(f"Text→PC R@1:   Guided={text_pc_guided['R@1']:.2f}%, Self={text_pc_self['R@1']:.2f}%, Gap={abs(r1_pc_diff):.2f}%")

# v4.4 target: gap < 15%
if abs(r1_brep_diff) < 10:
    print("\n✓ EXCELLENT: Self-grounding nearly matches text-guided (gap < 10%)!")
elif abs(r1_brep_diff) < 15:
    print("\n✓ GOOD: Self-grounding close to text-guided (gap < 15%)")
elif abs(r1_brep_diff) < 25:
    print(f"\n○ MODERATE: Self-grounding improving (gap: {abs(r1_brep_diff):.1f}%)")
else:
    print(f"\n△ NEEDS WORK: Self-grounding gap large (gap: {abs(r1_brep_diff):.1f}%, target: < 15%)")

In [ ]:
# Cell 11: Hierarchical Pattern Analysis (v4.4 Specific)

print("=" * 70)
print("HIERARCHICAL PATTERN ANALYSIS (v4.4 Topology)")
print("=" * 70)

if 'level_summaries' in embeddings:
    level_summaries = embeddings['level_summaries']  # (N, L, d)
    global_summary = embeddings.get('global_summary')  # (N, d)
    
    N, L, d = level_summaries.shape
    print(f"\nLevel summaries shape: {level_summaries.shape}")
    print(f"Number of BFS levels: {L}")
    
    # Analyze variance across levels
    print(f"\nPer-level statistics:")
    for l in range(L):
        level_l = level_summaries[:, l, :]  # (N, d)
        norm = level_l.norm(dim=-1).mean().item()
        var = level_l.var(dim=0).mean().item()  # Variance across samples
        print(f"  Level {l}: norm={norm:.3f}, var={var:.4f}")
    
    # Level-to-level correlation
    print(f"\nLevel-to-level correlation:")
    level_norms = F.normalize(level_summaries, dim=-1)  # (N, L, d)
    for l in range(min(L-1, 5)):  # First 5 level pairs
        # Correlation between adjacent levels
        corr = (level_norms[:, l] * level_norms[:, l+1]).sum(dim=-1).mean().item()
        print(f"  Level {l} ↔ Level {l+1}: {corr:.3f}")
    
    # Global summary relation to levels
    if global_summary is not None:
        print(f"\nGlobal summary relation to levels:")
        global_norm = F.normalize(global_summary, dim=-1)  # (N, d)
        for l in range(L):
            corr = (global_norm * level_norms[:, l]).sum(dim=-1).mean().item()
            print(f"  Global ↔ Level {l}: {corr:.3f}")
else:
    print("\nLevel summaries not available in model outputs.")
    print("This may indicate topology encoder is not producing level-wise features.")

In [ ]:
# Cell 12: Summary Table

print("\n" + "=" * 80)
print("EVALUATION SUMMARY - GFA v4.4 (Topology-Aware)")
print("=" * 80)

print(f"\n{'Metric':<40} {'Value':<15} {'Target':<15} {'Status'}")
print("-" * 80)

# Prepare metrics
metrics = [
    ('Query Alignment (KEY!)', f"{query_align_brep:.4f}", '≥0.85', query_align_brep >= 0.85),
    ('Self-cos BRep', f"{cos_brep:.4f}", '0.65-0.80', cos_brep >= 0.65),
    ('Self-cos PC', f"{cos_pc:.4f}", '0.65-0.80', cos_pc >= 0.65),
    ('Text→BRep R@1 (guided)', f"{text_brep_guided['R@1']:.2f}%", '≥65%', text_brep_guided['R@1'] >= 65),
    ('Text→BRep R@5 (guided)', f"{text_brep_guided['R@5']:.2f}%", '≥85%', text_brep_guided['R@5'] >= 85),
    ('Text→PC R@1 (guided)', f"{text_pc_guided['R@1']:.2f}%", '≥60%', text_pc_guided['R@1'] >= 60),
    ('Text→BRep R@1 (self)', f"{text_brep_self['R@1']:.2f}%", '40-60%', text_brep_self['R@1'] >= 40),
    ('Text→PC R@1 (self)', f"{text_pc_self['R@1']:.2f}%", '≥35%', text_pc_self['R@1'] >= 35),
    ('Gap (guided-self)', f"{abs(r1_brep_diff):.2f}%", '<15%', abs(r1_brep_diff) < 15),
    ('PC→BRep R@1 (guided)', f"{pc_brep_guided['R@1']:.2f}%", '≥80%', pc_brep_guided['R@1'] >= 80),
]

for metric, value, target, passed in metrics:
    status = '✓' if passed else '✗'
    print(f"{metric:<40} {value:<15} {target:<15} {status}")

passed_count = sum(1 for _, _, _, p in metrics if p)
print(f"\nPassed: {passed_count}/{len(metrics)}")

# Overall assessment
print("\n" + "-" * 80)
if passed_count == len(metrics):
    print("✓ ALL TARGETS MET - v4.4 topology encoding working as expected!")
    print("  Topology message passing successfully captures semantic features.")
elif passed_count >= len(metrics) - 2:
    print("○ MOSTLY PASSING - Minor adjustments may help")
elif query_align_brep >= 0.85:
    print("△ Query alignment excellent - Retrieval may need tuning")
elif cos_brep >= 0.65:
    print("△ Self-grounding meets target - Query alignment may need work")
elif cos_brep >= 0.50:
    print("△ Self-grounding improving - Continue training")
else:
    print("✗ Self-grounding needs improvement - Check topology encoder")

In [ ]:
# Cell 13: Embedding Visualization (UMAP)

try:
    from umap import UMAP
    
    # Sample for visualization
    sample_size = min(1000, len(z_text))
    idx = np.random.choice(len(z_text), sample_size, replace=False)
    
    # Combine embeddings
    all_emb = np.vstack([
        z_text[idx].numpy(),
        z_brep_guided[idx].numpy(),
        z_brep_self[idx].numpy(),
    ])
    
    labels = np.array(
        ['Text'] * sample_size + 
        ['BRep (Guided)'] * sample_size + 
        ['BRep (Self)'] * sample_size
    )
    
    # Reduce dimensions
    print("Running UMAP dimensionality reduction...")
    reducer = UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
    emb_2d = reducer.fit_transform(all_emb)
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Plot 1: All three modalities
    ax = axes[0]
    colors = {'Text': 'blue', 'BRep (Guided)': 'green', 'BRep (Self)': 'orange'}
    for label in ['Text', 'BRep (Guided)', 'BRep (Self)']:
        mask = labels == label
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], 
                   c=colors[label], label=label, alpha=0.5, s=10)
    ax.legend()
    ax.set_title('Embedding Space (UMAP)')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    # Plot 2: Guided vs Self with connections
    ax = axes[1]
    n_show = 50
    for i in range(n_show):
        guided_idx = sample_size + i
        self_idx = 2 * sample_size + i
        ax.plot([emb_2d[guided_idx, 0], emb_2d[self_idx, 0]], 
                [emb_2d[guided_idx, 1], emb_2d[self_idx, 1]], 
                'gray', alpha=0.3, linewidth=0.5)
    
    for label in ['BRep (Guided)', 'BRep (Self)']:
        mask = labels == label
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], 
                   c=colors[label], label=label, alpha=0.6, s=15)
    ax.legend()
    ax.set_title(f'Guided vs Self (Cosine: {cos_brep:.3f})')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    # Plot 3: Text vs BRep Self (text-free retrieval quality)
    ax = axes[2]
    for i in range(min(30, sample_size)):
        text_idx = i
        self_idx = 2 * sample_size + i
        ax.plot([emb_2d[text_idx, 0], emb_2d[self_idx, 0]], 
                [emb_2d[text_idx, 1], emb_2d[self_idx, 1]], 
                'purple', alpha=0.2, linewidth=0.5)
    
    for label in ['Text', 'BRep (Self)']:
        mask = labels == label
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], 
                   c=colors.get(label, 'purple'), label=label, alpha=0.6, s=15)
    ax.legend()
    ax.set_title(f'Text vs Self (Text-Free Retrieval)')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'embedding_visualization.png', dpi=150)
    print(f"Saved to {OUTPUT_DIR / 'embedding_visualization.png'}")
    plt.show()
    
except ImportError:
    print("UMAP not installed. Run: pip install umap-learn")

In [ ]:
# Cell 14: Self-Cosine Distribution Histogram

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# BRep self-cosine distribution
ax = axes[0]
ax.hist(cos_brep_all.numpy(), bins=50, alpha=0.7, color='green', edgecolor='black')
ax.axvline(x=cos_brep, color='red', linestyle='--', label=f'Mean: {cos_brep:.3f}')
ax.axvline(x=0.65, color='orange', linestyle=':', label='Target: 0.65')
ax.axvline(x=0.80, color='orange', linestyle=':', label='Target: 0.80')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Count')
ax.set_title('BRep Self-Grounding Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# PC self-cosine distribution
ax = axes[1]
ax.hist(cos_pc_all.numpy(), bins=50, alpha=0.7, color='blue', edgecolor='black')
ax.axvline(x=cos_pc, color='red', linestyle='--', label=f'Mean: {cos_pc:.3f}')
ax.axvline(x=0.65, color='orange', linestyle=':', label='Target: 0.65')
ax.axvline(x=0.80, color='orange', linestyle=':', label='Target: 0.80')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Count')
ax.set_title('PC Self-Grounding Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'self_cosine_distribution.png', dpi=150)
print(f"Saved to {OUTPUT_DIR / 'self_cosine_distribution.png'}")
plt.show()

In [ ]:
# Cell 15: Export Results to JSON

results = {
    'model': 'CLIP4CAD-GFA v4.4 (Topology-Aware)',
    'checkpoint': str(CHECKPOINT_PATH),
    'eval_date': datetime.now().isoformat(),
    'num_samples': len(val_dataset),
    'retrieval': {
        'text_brep_guided': text_brep_guided,
        'text_brep_self': text_brep_self,
        'text_pc_guided': text_pc_guided,
        'text_pc_self': text_pc_self,
        'pc_brep_guided': pc_brep_guided,
        'pc_brep_self': pc_brep_self,
    },
    'self_grounding': {
        'cos_brep': cos_brep,
        'cos_pc': cos_pc,
        'cos_brep_std': cos_brep_all.std().item(),
        'cos_pc_std': cos_pc_all.std().item(),
    },
    'query_alignment': {
        'brep': query_align_brep,
        'pc': query_align_pc,
    },
    'targets_met': {
        'query_align_target': query_align_brep >= 0.85,
        'cos_brep_target': cos_brep >= 0.65,
        'self_retrieval_target': text_brep_self['R@1'] >= 40,
        'gap_target': abs(r1_brep_diff) < 15,
    },
    'summary': {
        'passed': passed_count,
        'total': len(metrics),
        'key_innovations': [
            'Topology message passing via edge_to_faces',
            'BFS-level hierarchical pattern aggregation',
            'Multi-source query generation',
            'Contrastive query loss for instance matching',
        ]
    }
}

with open(OUTPUT_DIR / 'eval_results_v4_4.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {OUTPUT_DIR / 'eval_results_v4_4.json'}")

In [ ]:
# Cell 16: Cleanup

# Close dataset files
val_dataset.close()

# Clean memory
del embeddings, model
gc.collect()
torch.cuda.empty_cache()

print("\nEvaluation complete!")
print(f"Results saved to: {OUTPUT_DIR}")